In [1]:
import os
import math
import json
import warnings
import itertools

warnings.filterwarnings("ignore")

In [2]:
os.environ.setdefault("FINTORCH_BASE_TIME_FRAME", "60")

'60'

In [3]:
import numpy as np 
from tqdm.auto import tqdm
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

from fintorch.enum import TimeFrame
from fintorch.exchange import ONLINE_EXCHANGE
from fintorch.utils.preprocess import remove_price_dependency
from fintorch.utils.args import DefaultArgumentParser
from fintorch.utils.plot import draw_candlestick_plot, draw_labels, draw_predictions

In [4]:
string_args = [
    "--start-date", "2024-01-01",
    "--stop-date", "2024-06-01",
    "--time-frame", "60",
]

args = DefaultArgumentParser.parse(args=string_args)

In [5]:
sdf = ONLINE_EXCHANGE.future.data.get_candles_dataframe(symbol=args.symbol, time_frame=args.time_frame).copy()

sdf

,open,high,low,close,volume,trade
timestamp,,,,,,
1705395660,43032.7,43040.0,42985.6,43003.7,201.772,3323
1705395720,43004.5,43016.0,42960.0,42967.3,534.556,4093
1705395780,42967.3,42976.5,42954.2,42957.8,117.148,1847
1705395840,42957.9,42965.7,42928.9,42942.2,245.105,3502
1705395900,42942.3,42953.0,42919.0,42942.6,134.988,1928
...,...,...,...,...,...,...
1711406100,70419.7,70438.9,70406.9,70438.8,79.550,1522
1711406160,70438.7,70524.7,70438.7,70510.3,134.209,2770
1711406220,70510.3,70510.4,70437.2,70464.4,78.321,1687


In [6]:
LOOKAHEAD = 21
TRAIN_LENGTH = 100
SEQENCE_LENGTH = 128

y_hats_dict = {}
for stop_index in tqdm(range(TRAIN_LENGTH + SEQENCE_LENGTH + LOOKAHEAD, TRAIN_LENGTH + SEQENCE_LENGTH + LOOKAHEAD + 1000)):
    wdf = sdf.iloc[:stop_index] 

    # add features column
    features = ["roc", "tr"]
    wdf["roc"] = np.round((wdf.close / wdf.open - 1) * 10000, 2)
    wdf["tr"] = np.round((wdf.high / wdf.low - 1) * 10000, 2)
    
    # add label column
    sma = wdf.close.rolling(window=LOOKAHEAD, center=True).mean()
    fsma = sma.shift(periods=-LOOKAHEAD // 2)
    wdf["label"] = np.where(np.isnan(sma) | np.isnan(fsma), -1, sma < fsma)
    
    # prepare df
    wdf = wdf.dropna()
    wdf = wdf[-SEQENCE_LENGTH - TRAIN_LENGTH - LOOKAHEAD:]
    wdf["label"] = np.where(-1 == wdf.label, np.nan, wdf.label)
    wdf["label"] = np.where(wdf.label, 1, -1)
    
    # create smaple
    x, y = [], []
    for index in range(SEQENCE_LENGTH, len(wdf)):
        fdf = wdf.iloc[index - SEQENCE_LENGTH: index][features]
        fdf = (fdf - fdf.min()) / (fdf.max() - fdf.min())
        x.append(fdf.to_numpy())
        y.append(wdf.label.iloc[index])
        
    x = np.array(x).reshape(-1, SEQENCE_LENGTH * len(features))
    y = np.array(y)
    
    # print(x.shape)
    # print(y.shape)
    
    # split train and test sets
    train_x, train_y = x[:TRAIN_LENGTH], y[:TRAIN_LENGTH]
    test_x = x[-1:]
    
    # fit model
    model = LinearRegression(fit_intercept=False).fit(train_x, train_y)
        
    # predict last candle
    test_y_hat = model.predict(test_x)

    # append prediction
    timestamp = sdf.index[stop_index]
    y_hats_dict[timestamp] = test_y_hat.item()

  0%|          | 0/1000 [00:00<?, ?it/s]

In [11]:
df = sdf.copy()

df["sma"] = df.close.rolling(window=LOOKAHEAD, center=True).mean()
df["fsma"] = df.sma.shift(periods=-LOOKAHEAD // 2)
df["label"] = np.where(np.isnan(df.sma) | np.isnan(df.fsma), np.nan, df.sma < df.fsma)
df["label"] = np.where(df.label, 1, -1)
df = df.dropna()

y_hats = []
for index in range(len(df)):
    timestamp = df.index[index]
    y_hat = y_hats_dict.get(timestamp, np.nan)
    y_hats.append(y_hat)

df["y_hat"] = y_hats
df = df.dropna()

df["prediction"] = np.where(0 < np.abs(df.y_hat), np.sign(df.y_hat), np.nan)

print((df.label == df.prediction).sum() / len(df))

df

0.538


,open,high,low,close,volume,trade,sma,fsma,label,y_hat,prediction
timestamp,,,,,,,,,,,
1705410600,43112.9,43116.9,43080.2,43085.3,86.790,1610,43107.047619,43045.428571,-1,1.414269,1.0
1705410660,43085.4,43103.2,43070.0,43103.2,73.520,1597,43102.633333,43036.176190,-1,1.261782,1.0
1705410720,43103.3,43155.0,43103.3,43139.4,189.894,2470,43099.414286,43026.980952,-1,1.360584,1.0
1705410780,43139.5,43143.5,43118.3,43119.0,101.063,1583,43094.290476,43018.952381,-1,1.359380,1.0
1705410840,43118.9,43133.4,43106.4,43127.5,126.384,1845,43088.395238,43011.152381,-1,1.525933,1.0
...,...,...,...,...,...,...,...,...,...,...,...
1705470300,42837.7,42837.7,42827.1,42837.7,31.885,835,42829.876190,42807.523810,-1,0.409648,1.0
1705470360,42837.7,42837.7,42811.9,42812.0,63.914,1274,42826.280952,42809.409524,-1,0.314264,1.0
1705470420,42812.1,42830.0,42807.4,42830.0,54.228,1359,42820.742857,42810.195238,-1,0.608728,1.0


In [64]:
tdf = df.copy()


tdf["fmax"] = tdf.close.rolling(21).max().shift(-20)
tdf["fmin"] = tdf.close.rolling(21).min().shift(-20)
tdf["ltp"] =  tdf.fmax / tdf.open - 1
tdf["stp"] =  tdf.open / tdf.fmin - 1

tdf["tr"] = tdf.high / tdf.low - 1
tdf["roc"] = tdf.close / tdf.open - 1
tdf["atr"] = tdf.tr.rolling(window=10).mean()
tdf["aroc"] = np.abs(tdf.roc).rolling(window=10).mean()
tdf["bmin"] = tdf.close.rolling(20).min().shift(1)

tdf["buy"] = (1 == tdf.prediction) & (tdf.low <= tdf.bmin)
tdf["sell"] = -1 == tdf.prediction
tdf["tp"] = tdf.aroc
tdf = tdf.dropna()

print((tdf.label == tdf.prediction).sum() / len(tdf))
print(tdf[tdf.buy].ltp.mean())
print(tdf[tdf.sell].stp.mean())

print((tdf[tdf.buy].tp <= tdf[tdf.buy].ltp).sum() / tdf.buy.sum())
print((tdf[tdf.sell].tp <= tdf[tdf.sell].stp).sum() / tdf.sell.sum())

tdf

0.54375
0.002187790980060925
0.0015083691779090784
0.7909090909090909
0.8071570576540755


,open,high,low,close,volume,trade,sma,fsma,label,y_hat,...,ltp,stp,tr,roc,atr,aroc,bmin,buy,sell,tp
timestamp,,,,,,,,,,,,,,,,,,,,,
1705411800,42986.7,43000.0,42935.2,42947.5,387.354,5114,42974.585714,42941.761905,-1,0.970083,...,-0.000156,0.001813,0.001509,-0.000912,0.000955,0.000517,42961.7,True,False,0.000517
1705411860,42947.5,42962.4,42922.6,42933.0,329.927,4939,42967.319048,42936.866667,-1,1.014461,...,0.000757,0.001268,0.000927,-0.000338,0.000945,0.000448,42947.5,True,False,0.000448
1705411920,42933.1,42933.1,42905.7,42908.9,317.880,4481,42963.652381,42930.561905,-1,1.047839,...,0.001092,0.002403,0.000639,-0.000564,0.000928,0.000499,42933.0,True,False,0.000499
1705411980,42908.9,42949.6,42905.6,42946.3,194.000,3128,42959.914286,42923.528571,-1,1.111629,...,0.001657,0.003095,0.001026,0.000872,0.000906,0.000516,42908.9,True,False,0.000516
1705412040,42946.3,42962.0,42938.5,42950.4,119.132,2440,42958.114286,42917.780952,-1,1.085089,...,0.000785,0.003969,0.000547,0.000095,0.000834,0.000524,42908.9,False,False,0.000524
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1705469100,42877.7,42899.1,42871.0,42891.1,55.673,734,42876.428571,42865.376190,-1,-0.253001,...,0.000469,0.001331,0.000655,0.000313,0.000530,0.000346,42821.9,False,True,0.000346
1705469160,42891.2,42903.3,42891.2,42896.7,61.697,836,42879.695238,42862.200000,-1,-0.235598,...,0.000154,0.001850,0.000282,0.000128,0.000499,0.000310,42821.9,False,True,0.000310
1705469220,42896.8,42897.8,42880.2,42891.6,73.492,880,42881.190476,42858.266667,-1,-0.188624,...,0.000023,0.001981,0.000410,-0.000121,0.000509,0.000295,42821.9,False,True,0.000295


In [69]:
ENR = 1
TPR = 4
SLR = 1

#######################
###### backtest #######
#######################
bars_list, profits_list = [], []

order_index, order_price, entry_price, entry_index = None, None, None, None
for index in tqdm(range(len(tdf))):
    if entry_price is None:
        # reset order
        if order_index is not None and order_index + 10 < index:
            order_index = None
            order_price = None

        # set order
        if order_price is None and tdf.buy.iloc[index]:
            close_price = tdf.close.iloc[index]
            
            order_index = index
            order_price = round(close_price * (1 - tdf.aroc.iloc[index] * ENR), 2)

            # print(tdf.close.iloc[index])
            # print(order_price)
            # print()
            
        # check order status
        if order_price is not None and tdf.low.iloc[index] <= order_price:
            entry_index = index
            entry_price = order_price
            
            stop_loss = tdf.aroc.iloc[index] * SLR
            take_profit = tdf.aroc.iloc[index] * TPR
    
            stop_loss_price = round(entry_price * (1 - stop_loss - 0.0002 * 2), 2)
            take_profit_price = round(entry_price * (1 + take_profit + 0.0002 * 2), 2)
            
            # print(entry_price)
            # print(take_profit_price)
            # print(stop_loss_price)
            # print()
            
            order_index = None
            order_price = None
            
    else:
        profit = None
        if tdf.low.iloc[index] <= stop_loss_price:
            profit = -stop_loss
        elif take_profit_price <= tdf.high.iloc[index]:
            profit = take_profit

        if profit is not None:
            profits_list.append(profit)
            bars_list.append(index - entry_index)
            
            entry_index = None
            entry_price = None

  0%|          | 0/960 [00:00<?, ?it/s]

In [70]:
LEVERAGE = 50

bars = np.array(bars_list)
profits = np.array(profits_list) * LEVERAGE

days = len(tdf) * TimeFrame.MIN1 / TimeFrame.DAY1
min_bars = bars.min()
max_bars = bars.max()
mean_bars = bars.mean()

trades = len(profits)
daily_trades = len(profits) // days

net_profit = profits.sum()
compound_profit = (1 + profits).prod() - 1
daily_net_profit = net_profit / days
daily_compound_profit = (compound_profit + 1) ** (1 / days) - 1

wining_ratio = (0 < profits).sum() / len(profits) * 100

# print statements
print("{:<32}{}".format("Days", days))
print()
print("{:<32}{}".format("Min bars", min_bars))
print("{:<32}{:}".format("Max bars", max_bars))
print("{:<32}{:.2f}".format("Mean bars", mean_bars))
print()
print("{:<32}{}".format("Trades", trades))
print("{:<32}{}".format("Daily trades", daily_trades))
print()
print("{:<32}{:.2f}".format("Net profit", net_profit))
print("{:<32}{:.2f}".format("Compound profit", compound_profit))
print("{:<32}{:.2f}".format("Daily net profit", daily_net_profit))
print("{:<32}{:.2f}".format("Daily compound profit", daily_compound_profit))
print()
print("{:<32}{:.2f}%".format("Wining ratio", wining_ratio))


Days                            0.6666666666666666

Min bars                        1
Max bars                        20
Mean bars                       6.19

Trades                          32
Daily trades                    48.0

Net profit                      0.85
Compound profit                 1.11
Daily net profit                1.28
Daily compound profit           2.06

Wining ratio                    40.62%
